In [ ]:
# Install Required Libraries
!pip install -q pinecone
!pip install -q langchain-pinecone
!pip install -q --upgrade langchain
!pip install -q --upgrade langchain-community
!pip install -q langchain-groq
!pip install -q langchain-classic
!pip install -q sentence-transformers
!pip install -q gradio
!pip install -q tiktoken
!pip install -q pypdf
print("libraries install successfully!")

libraries install successfully!


In [ ]:
# IMPORT ALL REQUIRED LIBRARIES
from google.colab import files
import os
from getpass import getpass
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore
from langchain_groq import ChatGroq
import gradio as gr
from langchain_classic.chains import RetrievalQA
print("All libraries imported successfully!")

All libraries imported successfully!


In [ ]:
from google.colab import files

uploaded = files.upload()

print("File uploaded successfully!")

Saving benazir_kafaalat_data.txt to benazir_kafaalat_data (3).txt
File uploaded successfully!


In [ ]:
# Load Text File
file_name = list(uploaded.keys())[0]

# Read text file
with open(file_name, "r", encoding="latin-1") as file:
    data = file.read()

print("Data Loaded Successfully!\n")

print(data[:1000])  # Preview first 1000 characters

Data Loaded Successfully!

# Benazir Kafaalat Programme - Official Data for RAG Chatbot
## Program Overview
 The Unconditional Cash Transfers (UCT) programme also known as Benazir Kafaalat Programme is the core programme of BISP. It was initiated in the year 2008. Since inception, the UCT/ Kafaalat initiative has grown to the extent that it is now the largest single cash transfer programme in Pakistan's history. 
BISP is currently providing cash assistance to around 9 million families under Benazir Kafaalat programme. Starting from Rs. 3,000/-per beneficiary per quarter, the present Benazir Kafaalat stipend is Rs. 8,500/- per beneficiary per quarter.
## Eligibility Criteria:
### Standard Eligibility (Based on PMT Score)
Beneficiaries of Benazir Kafaalat Programme are identified/selected through scientific mode of Proxy Means Test (PMT) through National Socio Economic Registry (NSER) survey. The welfare status of a household is determined on a scale between 0-100 of the PMT. The PMT

In [ ]:
# Split Text into Chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

texts = text_splitter.split_text(data)

print("Total Chunks:", len(texts))

# Preview first chunk
print("\nFirst Chunk:\n")
print(texts[0])

Total Chunks: 13

First Chunk:

# Benazir Kafaalat Programme - Official Data for RAG Chatbot
## Program Overview
 The Unconditional Cash Transfers (UCT) programme also known as Benazir Kafaalat Programme is the core programme of BISP. It was initiated in the year 2008. Since inception, the UCT/ Kafaalat initiative has grown to the extent that it is now the largest single cash transfer programme in Pakistan's history.


In [ ]:
# Create HuggingFace Embeddings
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embeddings model loaded successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embeddings model loaded successfully!


In [ ]:
# Enter Groq API Key
os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API Key: ")

print("API Key Added Successfully!")

Enter your Groq API Key: ··········
API Key Added Successfully!


In [ ]:
# Load Groq LLM
llm = ChatGroq(
    model_name="llama3-8b-8192",
    temperature=0
)

print("Groq Llama3 loaded successfully!")

Groq Llama3 loaded successfully!


In [ ]:
# Pinecone API Key
os.environ["PINECONE_API_KEY"] = getpass("Enter Pinecone API Key: ")

Enter Pinecone API Key: ··········


In [ ]:
# Pinecone Initialize

pc = Pinecone(
    api_key=os.environ["PINECONE_API_KEY"]
)

index_name = "benazir-rag"

# Create index if not exists
if index_name not in pc.list_indexes().names():

    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

print("Pinecone index ready")


Pinecone index ready


In [ ]:
# Store Embeddings in Pinecone
vectorstore = PineconeVectorStore.from_texts(
    texts,
    embedding_model,
    index_name=index_name
)

print("Data stored in Pinecone")

Data stored in Pinecone


In [ ]:
# Create retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

print("Retriever created successfully!")

Retriever created successfully!


In [ ]:
# Create Retrieval QA Chain
llm = ChatGroq(temperature=0, groq_api_key=os.environ["GROQ_API_KEY"], model_name="llama-3.1-8b-instant")
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

# Create QA chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff"
)

print("✅ Retrieval QA Chain Ready!")

✅ Retrieval QA Chain Ready!


In [ ]:
# Test Query
query = "What is the PMT score for Benazir Kafaalat eligibility?"

response = qa_chain.run(query)

print("Question:")
print(query)

print("\nAnswer:")
print(response)

Question:
What is the PMT score for Benazir Kafaalat eligibility?

Answer:
The current PMT cut-off score for eligibility under the Benazir Kafaalat Programme is 32.


In [ ]:
# Urdu + English Chat Function
def chatbot(question):
    try:
        response = qa_chain.run(question)
        return response

    except Exception as e:
        return f"Error: {str(e)}"

In [ ]:
# Create Gradio Interface
interface = gr.Interface(
    fn=chatbot,
    inputs=gr.Textbox(
        lines=8,
        placeholder="Ask your question in Urdu or English..."
    ),
    outputs=gr.Textbox(lines=8),
    title="Benazir Kafaalat Eligibility Assistant",
    description="""
    Ask questions about:
    - Eligibility
    - PMT score
    - Registration
    - Documents
    - Stipend
    - 8171 SMS service
    """
)

interface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2499f096503cacf442.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
